## Inverse Kinemaics - Position and Rotation together
- Stack two jacobians (position and rotation) to reach both p&r goal 

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [ ]:
model_path = "../assets/ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

Get Mujoco Jacobian

In [3]:
def get_jac_body_name(body_name=None):

    # initialize positional & rotational jacobian
    Jacobian_p = np.zeros((3,model.nu))
    Jacobian_r = np.zeros((3,model.nu))

    # get jacobian of end-effector
    mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
    return Jacobian_p, Jacobian_r

### Function: calculate inverse jacobian

In [4]:
def get_invjac_name(body_name, method='svd', sigma_threshold=0.001, damping=1.0):
    # get inversed jacobian including both position & rotation

    jacobian_p, jacobian_r = get_jac_body_name(body_name)
    jacobian_full = np.vstack([jacobian_p, jacobian_r])
    

    if method=='svd':
        # get inverse jacobian with Singular Value Decomposition
        U, Sigma, V_T = np.linalg.svd(jacobian_full, compute_uv=True)
        print(f"shape: {Sigma.shape}")

        # past implementation - not good
        # Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)

        # suppress singularities modifying sigma
        Sigma_clipped_rev = np.zeros_like(Sigma)
        for i, value in enumerate(Sigma):
            if Sigma[i] < sigma_threshold:
                Sigma_clipped_rev[i] = 0
            else:
                Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((model.nu,6)) # positional dimension = 3, dof = model.nu
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        J_inverse = V_T.T @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        pass
    
    return J_inverse

# mujoco.mj_forward(model, data)
# J_inverse = get_invjac_name("wrist_3_link")
# J_inverse

### Calculating Error: position & rotation
- position error: difference of global position
- orientation error: error = rotation_goal and rotation_current matrix multiplication

In [5]:
# angle convert functions

def q_to_rmat(wxyz):
    """ WXYZ order!! """
    # initialize matrix
    value = np.zeros(9)
    
    # built-in function
    mujoco.mju_quat2Mat(value, wxyz)
    value = value.reshape(3,3)

    return value


def rpy_to_rmat(rpy=[0.0]*3):

    roll, pitch, yaw = rpy[0], rpy[1], rpy[2]
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw), np.sin(yaw)

    R = np.array([
        [cy * cp, cy * sp * sr - sy * cr, cy * sp * cr + sy * sr],
        [sy * cp, sy * sp * sr + cy * cr, sy * sp * cr - cy * sr],
        [-sp,     cp * sr,                cp * cr]
    ])
    return R


def rmat_to_rpy(R):

    # check for gimbal lock
    if abs(R[2, 0]) != 1:
        pitch = -np.arcsin(R[2, 0])
        roll = np.arctan2(R[2, 1] / np.cos(pitch), R[2, 2] / np.cos(pitch))
        yaw = np.arctan2(R[1, 0] / np.cos(pitch), R[0, 0] / np.cos(pitch))
    else:
        print("gimbal lock occurred, setting yaw to 0..")
        # Gimbal lock: pitch = ±90°
        yaw = 0  # arbitrary
        if R[2, 0] == -1:
            pitch = np.pi / 2
            roll = yaw + np.arctan2(R[0, 1], R[0, 2])
        else:
            pitch = -np.pi / 2
            roll = -yaw + np.arctan2(-R[0, 1], -R[0, 2])
    return np.array([roll, pitch, yaw])



rpy = [0.3, 0.2, 0.1]
R = rpy_to_rmat(rpy)
print("Rotation matrix:\n", R)

rpy = rmat_to_rpy(R)
print("Recovered RPY:", rpy)


Rotation matrix:
 [[ 0.97517033 -0.03695701  0.21835066]
 [ 0.0978434   0.95642509 -0.27509585]
 [-0.19866933  0.28962948  0.93629336]]
Recovered RPY: [0.3 0.2 0.1]


In [6]:
# get orientation error

body_name = "wrist_3_link"

# step once to get orientation error
mujoco.mj_forward(model, data)

# 1. get error with difference
rpy_goal = [0.2]* 3
r_curr = rmat_to_rpy(data.body(body_name).xmat.reshape(3,3))

r_diff = rpy_goal - r_curr
r_diff

array([-2.94159265,  0.2       ,  1.77079633])

In [7]:

# 2. get error with rmat multiplication
rpy_goal = [0.2]* 3
r_goal = rpy_to_rmat(rpy_goal)
r_curr = data.body(body_name).xmat.reshape(3,3)

rmat_diff = r_goal @ r_curr.T

r_diff = rmat_to_rpy(rmat_diff)
r_diff


array([ 2.93763569,  0.19596094, -1.41104684])

In [8]:
"""
difference
- roll and yaw sign is different..
"""

'\ndifference\n- roll and yaw sign is different..\n'

### MAIN loop: calculate error & update with forward

In [30]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# initialize robot
init_qpos = [3.14, -0.8, -2.5, -2.2, 0.0, 0.0]

mujoco.mj_resetData(model, data)
data.qpos = init_qpos
mujoco.mj_forward(model, data) # first forward to get jacobian with no error

# goal position
goal_p = [0.4, 0.4, 0.2]
# goal orientation
rpy_goal = [3.14, 0.0, 0.0]
r_goal = rpy_to_rmat(rpy_goal)

body_name = "wrist_3_link"

# scale down error - IK slow down
alpha_p = 0.02
alpha_r = 0.02 # more damping on rotation

while True:
    if viewer.is_alive:

        # get inverse jacobian & unit error vector
        J_inverse = get_invjac_name(body_name=body_name, method='svd')

        # get error: position & rotation
        error_p = goal_p - data.body(body_name).xpos.copy()
        r_curr = data.body(body_name).xmat.reshape(3,3).T
        error_rmat = r_goal @ r_curr.T
        error_r = rmat_to_rpy(error_rmat)
        # error_r = np.zeros(3)
        # mujoco.mju_mat2vel(error_r, error_rmat.flatten(), 1.0)

        error = np.hstack([alpha_p * error_p, alpha_r * error_r]).T

        # print(f"size check - J inverse: {J_inverse.shape}, error: {error.shape}")
        dq = J_inverse @ error
        print(f"dq: {dq}")
        data.qpos += dq
        # print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

        mujoco.mj_forward(model, data)

        # print(f"current body pos: {data.body(body_name).xpos}")

        if np.linalg.norm(error_p) < 0.05:
            alpha_p = 0.01
            print("within position threshold")
        if np.linalg.norm(error_r) < 0.05:
            alpha_r = 0.0005
            print("within rotation threshold")

        print(f"positional, rotational difference: \n pos: {error_p}, rot: {error_r}")

        # adjust alpha value through reaching

        # terminalize
        if np.linalg.norm(goal_p - data.body(body_name).xpos) < 0.01 and np.linalg.norm(rpy_goal - rmat_to_rpy(data.body(body_name).xmat.reshape(3,3))) < 0.01:
            print("IK done.")
            break

        viewer.render()

    else:
        break

# close
viewer.close()

shape: (6,)
dq: [-0.01653827 -0.03131168  0.00231927  0.01833211  0.0215497   0.01063534]
positional, rotational difference: 
 pos: [ 0.26574288  0.23866787 -0.13517305], rot: [-3.57364035e-06  7.81592656e-01 -1.56920619e+00]
shape: (6,)
dq: [-0.03070033  0.08248965 -0.1236583   0.56892415  0.01072866 -0.52773922]
positional, rotational difference: 
 pos: [ 0.26296675  0.23393625 -0.13235023], rot: [ 0.02110936  0.78134093 -1.52234051]
shape: (6,)
dq: [-0.02948574 -0.04192813 -0.03088495  0.19746689  0.01511868 -0.12460873]
positional, rotational difference: 
 pos: [ 0.25801502  0.2398804  -0.13678641], rot: [ 0.04367261  0.78097466 -1.48253304]
shape: (6,)
dq: [-0.02678098 -0.04607733  0.00044233  0.05195387  0.01578762 -0.0062346 ]
positional, rotational difference: 
 pos: [ 0.25276919  0.2366563  -0.13422875], rot: [ 0.06582014  0.7803876  -1.43916912]
shape: (6,)
dq: [-0.02424553 -0.04157251  0.00841517  0.01055119  0.01604433  0.02276182]
positional, rotational difference: 
 pos: 

Error debugging
- robot shaking: due to different goal reaching
    - roll difference almost 3.14 -> norm calculation and rotation calculation different
- one more possibility: wrapping - spinning long way
    - method: convert error matrix into axis angle velocity vector